In [1]:
!pip install datasets trl peft accelerate loralib evaluate  bitsandbytes lora -q

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForQuestionAnswering, AutoTokenizer, DataCollatorWithPadding,
    Trainer, TrainingArguments, BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig, PeftModel, PeftConfig
from sklearn.metrics.pairwise import cosine_similarity
import loralib as lora
from transformers.trainer_utils import IntervalStrategy

# 1. Load dataset SQuAD 2.0

In [3]:
# 1. Load dataset SQuAD 2.0
dataset = load_dataset("squad_v2")

dataset

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [4]:
dataset['train'][0]

{'id': '56be85543aeaaa14008c9063',
 'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': 'When did Beyonce start becoming popular?',
 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}

# 2. Tokenize dữ liệu

In [5]:
# 2. Tokenize dữ liệu
model_name = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
def preprocess_data(examples):
  try:
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation=True,
        padding="max_length",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = inputs["input_ids"][i]
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start = 0
            token_end = 0
            for idx, (offset_start, offset_end) in enumerate(offsets):
                if offset_start <= start_char < offset_end:
                    token_start = idx
                if offset_start < end_char <= offset_end:
                    token_end = idx
                    break

            start_positions.append(token_start)
            end_positions.append(token_end)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs
  except Exception as e:
    print(f"Error processing example: {examples['answers']}")
    print(f"error in index: ", i)
    print(f"Error message: {e}")
    return {}

In [7]:
tokenized_dataset = dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

In [15]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 131754
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 12134
    })
})

In [16]:
print('test:' , tokenized_dataset['train'][0]['start_positions'])
print('test:' , tokenized_dataset['train'][0]['end_positions'])

test: 75
test: 78


# 3. Config Lora và tiến hành Fine tuning BERT

In [17]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="QUESTION_ANS"
)

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

model = get_peft_model(model, lora_config)
model

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForQuestionAnswering(
  (base_model): LoraModel(
    (model): BertForQuestionAnswering(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear

In [18]:
from sklearn.metrics import accuracy_score

def compute_metrics(p):
    try:
        start_logits, end_logits = p.predictions
        start_labels, end_labels = p.label_ids

        start_preds = torch.argmax(torch.tensor(start_logits), dim=-1)
        end_preds = torch.argmax(torch.tensor(end_logits), dim=-1)

        start_accuracy = (start_preds == torch.tensor(start_labels)).float().mean().item()
        end_accuracy = (end_preds == torch.tensor(end_labels)).float().mean().item()

        return {
          "start_accuracy": start_accuracy,
          "end_accuracy": end_accuracy,
        }
    except Exception as e:
        print(f"Error in compute_metrics: {e}")
        return {
            "start_accuracy": 0.0,
            "end_accuracy": 0.0,
        }

In [19]:
# Thiết lập cấu hình cho fine-tuning
training_args = TrainingArguments(
    output_dir="./bert_lora_fine_tuned",
    num_train_epochs=1,
    per_device_train_batch_size=110,
    per_device_eval_batch_size=110,
    gradient_accumulation_steps=1,
    save_strategy="steps",
    save_steps=500,
    logging_dir='./logs',
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    dataloader_num_workers=2,
    eval_strategy="steps",
    eval_steps=500,
    report_to="none",
    load_best_model_at_end=True,
)

In [20]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [21]:
# Fine-tune mô hình
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning completed!")

Starting fine-tuning...


Step,Training Loss,Validation Loss,Start Accuracy,End Accuracy
500,2.260500,1.840311,0.489286,0.481704
1000,2.080500,1.771657,0.477831,0.465799


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Fine-tuning completed!


# 4. Lưu model và tiến hành test

In [22]:
trainer.save_model("./bert_lora_fine_tuned")
tokenizer.save_pretrained("./bert_lora_fine_tuned")

('./bert_lora_fine_tuned/tokenizer_config.json',
 './bert_lora_fine_tuned/special_tokens_map.json',
 './bert_lora_fine_tuned/vocab.txt',
 './bert_lora_fine_tuned/added_tokens.json',
 './bert_lora_fine_tuned/tokenizer.json')

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
model.save_pretrained("/content/drive/MyDrive/0. My_project/fine_tuned_models/fine_tuned_bert")
tokenizer.save_pretrained("/content/drive/MyDrive/0. My_project/fine_tuned_models/fine_tuned_bert")
print("Model saved successfully to Google Drive!")

Model saved successfully to Google Drive!


In [43]:
trainer.model

PeftModelForQuestionAnswering(
  (base_model): LoraModel(
    (model): BertForQuestionAnswering(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear

In [58]:
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("./bert_lora_fine_tuned")

qa_pipeline = pipeline(
    "question-answering",
    model=trainer.model,
    tokenizer=tokenizer
)

context = '''Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".'''
question = "When did Beyonce start becoming popular?"

result = qa_pipeline(question=question, context=context)
print(result)

Device set to use cuda:0
The model 'PeftModelForQuestionAnswering' is not supported for question-answering. Supported models are ['AlbertForQuestionAnswering', 'BartForQuestionAnswering', 'BertForQuestionAnswering', 'BigBirdForQuestionAnswering', 'BigBirdPegasusForQuestionAnswering', 'BloomForQuestionAnswering', 'CamembertForQuestionAnswering', 'CanineForQuestionAnswering', 'ConvBertForQuestionAnswering', 'Data2VecTextForQuestionAnswering', 'DebertaForQuestionAnswering', 'DebertaV2ForQuestionAnswering', 'DistilBertForQuestionAnswering', 'ElectraForQuestionAnswering', 'ErnieForQuestionAnswering', 'ErnieMForQuestionAnswering', 'FalconForQuestionAnswering', 'FlaubertForQuestionAnsweringSimple', 'FNetForQuestionAnswering', 'FunnelForQuestionAnswering', 'GPT2ForQuestionAnswering', 'GPTNeoForQuestionAnswering', 'GPTNeoXForQuestionAnswering', 'GPTJForQuestionAnswering', 'IBertForQuestionAnswering', 'LayoutLMv2ForQuestionAnswering', 'LayoutLMv3ForQuestionAnswering', 'LEDForQuestionAnswering', 

{'score': 0.023111306130886078, 'start': 64, 'end': 81, 'answer': 'September 4, 1981'}


In [59]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline, logging
from peft import PeftModel, PeftConfig
logging.set_verbosity_info()
logging.set_verbosity_warning()

model_name = "bert-base-uncased"
model_fine_tuned = "./bert_lora_fine_tuned"

model = AutoModelForQuestionAnswering.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_fine_tuned)

config = PeftConfig.from_pretrained(model_fine_tuned)
model = PeftModel.from_pretrained(model, model_fine_tuned, config=config)

qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

context = '''The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was designed by the engineer Gustave Eiffel and completed in 1889 as the entrance arch to the 1889 World's Fair. At the time of its construction, it was the tallest man-made structure in the world.'''
question = "Who designed the Eiffel Tower?"
result = qa_pipeline(question=question, context=context)
print(result)

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
The model 'PeftModelForQuestionAnswering' is not supported for question-answering. Supported models are ['AlbertForQuestionAnswering', 'BartForQuestionAnswering', 'BertForQuestionAnswering', 'BigBirdForQuestionAnswering', 'BigBirdPegasusForQuestionAnswering', 'BloomForQuestionAnswering', 'CamembertForQuestionAnswering', 'CanineForQuestionAnswering', 'ConvBertForQuestionAnswering', 'Data2VecTextForQuestionAnswering', 'DebertaForQuestionAnswering', 'DebertaV2ForQuestionAnswering', 'DistilBertForQuestionAnswering', 'ElectraForQuestionAnswering', 'ErnieForQuestionAnswering', 'ErnieMForQuestionAnswering', 'FalconForQuestionAnswering', 'FlaubertForQuestionAnsweringSimple', 'FNetForQue

{'score': 0.3460879325866699, 'start': 120, 'end': 134, 'answer': 'Gustave Eiffel'}


# 5. Merge base_model vs phần Adapter sau khi fine tune

In [62]:
model_name = "bert-base-uncased"
model_fine_tuned = "./bert_lora_fine_tuned"

model = AutoModelForQuestionAnswering.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_fine_tuned)

model = PeftModel.from_pretrained(model, model_fine_tuned)

model = model.merge_and_unload()

model.save_pretrained("./final_fine-tuned_merged")
tokenizer.save_pretrained("./final_fine-tuned_merged")

model.save_pretrained("/content/drive/MyDrive/0. My_project/fine_tuned_models/fine_tuned_bert_merged")
tokenizer.save_pretrained("/content/drive/MyDrive/0. My_project/fine_tuned_models/fine_tuned_bert_merged")

print("Model saved successfully!")

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model saved successfully!


In [61]:
merged_model = AutoModelForQuestionAnswering.from_pretrained("./final_fine-tuned_merged")
merged_tokenizer = AutoTokenizer.from_pretrained("./final_fine-tuned_merged")

qa_pipeline = pipeline(
    "question-answering",
    model=merged_model,
    tokenizer=merged_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

context = '''The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was designed by the engineer Gustave Eiffel and completed in 1889 as the entrance arch to the 1889 World's Fair. At the time of its construction, it was the tallest man-made structure in the world.'''
question = "Who designed the Eiffel Tower?"
result = qa_pipeline(question=question, context=context)
print(result)

Device set to use cuda:0


{'score': 0.14597387611865997, 'start': 120, 'end': 134, 'answer': 'Gustave Eiffel'}
